Import delle librerie necessarie

In [30]:
import os
from PIL import Image
from sklearn.model_selection import train_test_split  # aggiunto import
import torch
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from torchvision.transforms import InterpolationMode


torch.cuda.empty_cache()

# Verifica se CUDA è disponibile
if torch.cuda.is_available():
    device = torch.device('cuda')  # Configura per utilizzare la GPU
    print(f"Utilizzo GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')  # Fallback alla CPU
    print("CUDA non disponibile, utilizzo CPU")

# Esempio di spostamento di un tensore sulla GPU
x = torch.randn(3, 3)  # Tensore su CPU
x = x.to(device)       # Sposta il tensore su GPU (se disponibile)
print(x)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No CUDA device")
from torchvision import transforms
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

Utilizzo GPU: NVIDIA GeForce RTX 5070 Ti
tensor([[-1.1950, -0.6834, -0.7126],
        [ 0.5212,  0.1160,  0.9322],
        [ 2.3368, -0.8478, -1.9622]], device='cuda:0')
True
NVIDIA GeForce RTX 5070 Ti


Import del training set

In [12]:

train_dir = 'train'
X_images = []
y_labels = []

for folder in sorted(os.listdir(train_dir)):
    folder_path = os.path.join(train_dir, folder)
    if os.path.isdir(folder_path):
        # Cerca file .jpg e .png nella sottocartella
        jpg_files = [f for f in os.listdir(folder_path) if f.endswith('.jpg')]
        png_files = [f for f in os.listdir(folder_path) if f.endswith('.png')]
        if jpg_files and png_files:
            X_path = os.path.join(folder_path, jpg_files[0])
            y_path = os.path.join(folder_path, png_files[0])
            X_images.append(Image.open(X_path))
            y_labels.append(Image.open(y_path))

 Divisione del training set in train e validation

In [31]:
# Suddivisione 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_images, y_labels, test_size=0.2, random_state=42, shuffle=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizzo dispositivo: {device}")

# Conversione delle immagini PIL in tensori e spostamento su GPU
transform = transforms.Compose([
    transforms.Resize((256, 136)),  # oppure (512, 272) se vuoi più dettaglio
    transforms.ToTensor()
])
transform_lbl = transforms.Compose([
    transforms.Resize((256, 136), interpolation=InterpolationMode.NEAREST),  # per le label
    transforms.ToTensor()
])

X_train = [transform(img).to(device) for img in X_train]
y_train = [transform(lbl).to(device) for lbl in y_train]
X_val = [transform(img).to(device) for img in X_val]
y_val = [transform(lbl).to(device) for lbl in y_val]

print(f"Numero immagini in X_train: {len(X_train)}")
print(f"Numero immagini in X_val: {len(X_val)}")

batch_size = 4

# Stack dei tensori per creare dataset compatibili con DataLoader
X_train_tensor = torch.stack(X_train)
y_train_tensor = torch.stack(y_train)
X_val_tensor = torch.stack(X_val)
y_val_tensor = torch.stack(y_val)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Numero batch in train_loader: {len(train_loader)}")
print(f"Numero batch in val_loader: {len(val_loader)}")


Utilizzo dispositivo: cuda
Numero immagini in X_train: 744
Numero immagini in X_val: 187
Numero batch in train_loader: 186
Numero batch in val_loader: 47


Training e validation del modello

In [32]:
import torch.nn as nn
import torch.optim as optim

# Definizione della rete convoluzionale
class SimpleSegmentationCNN(nn.Module):
    def __init__(self, num_classes=9):
        super(SimpleSegmentationCNN, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, num_classes, kernel_size=2, stride=2)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

# Funzione per convertire le label RGB in classi (0-8)
def rgb_to_class(label_tensor):
    # label_tensor: (B, 3, H, W) valori [0,1]
    color_map = {
        (1, 1, 1): 0, # Background (255,255,255)
        (0, 0, 1): 1, # Sky (1,8,255)
        (0.611, 0.298, 0.117): 2, # Rough Trail (156,76,30)
        (0.698, 0.690, 0.6): 3, # Smooth Trail (178,176,153)
        (0.502, 1, 0): 4, # Traversable grass (128,255,0)
        (0.157, 0.314, 0): 5, # High Vegetable (40,80,0)
        (0, 0.627, 0): 6, # Non Traversable Low Vegetation (0,160,0)
        (1, 0, 0.502): 7, # Puddle (255,0,128)
        (1, 0, 0): 8 # Obstacle (255,0,0)
    }
    # Arrotonda i valori per evitare errori di float
    label_tensor = torch.round(label_tensor * 255) / 255
    B, C, H, W = label_tensor.shape
    label_class = torch.zeros((B, H, W), dtype=torch.long, device=label_tensor.device)
    for rgb, idx in color_map.items():
        mask = (label_tensor[0].permute(1,2,0) == torch.tensor(rgb, device=label_tensor.device)).all(dim=-1)
        label_class[0][mask] = idx
    return label_class

# Istanziamento modello, loss e ottimizzatore
model = SimpleSegmentationCNN(num_classes=9).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

epochs = 5  # Puoi aumentare per un training reale
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    model.train()
    train_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        # Converti le label RGB in classi
        labels_class = rgb_to_class(labels)
        # Stampa esempio di assegnazione pixel-classe-colore
        if epoch == 0 and batch_idx == 0:
            # Prendi il primo pixel della prima immagine del batch
            pixel_rgb = inputs[0, :, 0, 0].cpu().numpy()  # (3,)
            pixel_rgb_img = (pixel_rgb * 255).astype(int)
            pixel_label_rgb = labels[0, :, 0, 0].cpu().numpy()  # (3,)
            pixel_label_rgb_img = (pixel_label_rgb * 255).astype(int)
            pixel_label_class = labels_class[0, 0, 0].item()
            print(f"Esempio assegnazione: pixel immagine RGB {pixel_rgb_img} -> classe {pixel_label_class} (colore label {pixel_label_rgb_img})")
        optimizer.zero_grad()
        outputs = model(inputs)
        # outputs: (B, 9, H, W), labels_class: (B, H, W)
        loss = criterion(outputs, labels_class)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        if (batch_idx+1) % 5 == 0 or (batch_idx+1) == len(train_loader):
            print(f"  [Batch {batch_idx+1}/{len(train_loader)}] Loss: {loss.item():.4f}")
    print(f"  Training Loss: {train_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(val_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            labels_class = rgb_to_class(labels)
            outputs = model(inputs)
            loss = criterion(outputs, labels_class)
            val_loss += loss.item()
    print(f"  Validation Loss: {val_loss/len(val_loader):.4f}")
    torch.cuda.empty_cache()

Epoch 1/5
Esempio assegnazione: pixel immagine RGB [18 28 27] -> classe 0 (colore label [7])
  [Batch 5/186] Loss: 2.1305
  [Batch 10/186] Loss: 1.7856
  [Batch 15/186] Loss: 1.2056
  [Batch 20/186] Loss: 0.4877
  [Batch 25/186] Loss: 0.1948
  [Batch 30/186] Loss: 0.0179
  [Batch 35/186] Loss: 0.0063
  [Batch 40/186] Loss: 0.0007
  [Batch 45/186] Loss: 0.0004
  [Batch 50/186] Loss: 0.0003
  [Batch 55/186] Loss: 0.0002
  [Batch 60/186] Loss: 0.0002
  [Batch 65/186] Loss: 0.0001
  [Batch 70/186] Loss: 0.0001
  [Batch 75/186] Loss: 0.0001
  [Batch 80/186] Loss: 0.0001
  [Batch 85/186] Loss: 0.0000
  [Batch 90/186] Loss: 0.0000
  [Batch 95/186] Loss: 0.0000
  [Batch 100/186] Loss: 0.0000
  [Batch 105/186] Loss: 0.0000
  [Batch 110/186] Loss: 0.0001
  [Batch 115/186] Loss: 0.0000
  [Batch 120/186] Loss: 0.0000
  [Batch 125/186] Loss: 0.0002
  [Batch 130/186] Loss: 0.0000
  [Batch 135/186] Loss: 0.0000
  [Batch 140/186] Loss: 0.0000
  [Batch 145/186] Loss: 0.0000
  [Batch 150/186] Loss: 0.00

# Salvataggio del modello addestrato

In [19]:
model_path = "model_segmentation.pth"
torch.save(model.state_dict(), model_path)
print(f"Modello salvato in {model_path}")

Modello salvato in model_segmentation.pth


# Caricamento del modello addestrato

In [21]:
def load_model():
    model = SimpleSegmentationCNN(num_classes=9)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    print(f"Modello caricato da {model_path} e pronto all'uso.")
    return model

# Predizione delle immagini di test

In [23]:
import torch

def predict(model, X):
    """
    Esegue la predizione su una singola immagine numpy (H, W, 3) o PIL.Image.
    Restituisce la maschera predetta come array numpy (H, W) di classi.
    """
    model.eval()
    with torch.no_grad():
        # Se X è PIL.Image, convertilo in numpy
        if hasattr(X, 'convert'):
            X = np.array(X)
        # Pre-processing: (H, W, 3) -> (1, 3, H, W), float, normalizzato [0,1]
        X_tensor = torch.from_numpy(X).permute(2, 0, 1).unsqueeze(0).float() / 255.0
        X_tensor = X_tensor.to(device)
        # Forward
        Y = model(X_tensor)
        # Post-processing: (1, 9, H, W) -> (H, W)
        Y = Y.squeeze(0).argmax(dim=0).cpu().numpy().astype(np.uint8)
    return Y


def compute_iou(mask1, mask2, label):
  intersection = np.sum((mask1 == label) & (mask2 == label))
  union = np.sum((mask1 == label) | (mask2 == label))
  if union == 0:
    return np.nan
  return intersection / union
def compute_all_iou(mask1, mask2, num_labels=8):
  iou_scores = np.zeros((num_labels))
  for label in range(num_labels):
    iou = compute_iou(mask1, mask2, label+1) # we skip the background label
    iou_scores[label] = iou
  return iou_scores


# Run YOUR LOAD_MODEL FUNCTION
model = load_model()

# Main loop
test_dir = "./train"  # we will change this path with that of the private test set directory
samples = os.listdir(test_dir)
IOUs = np.zeros((len(samples), 8))
verbose = False

for i, subdir in tqdm(enumerate(samples), desc="Processing samples"):
    subdir_path = os.path.join(test_dir, subdir)

    if os.path.isdir(subdir_path):
        # Get the data paths
        rgb_path = os.path.join(subdir_path, 'rgb.jpg')
        labels_path = os.path.join(subdir_path, 'labels.png')

        if os.path.exists(rgb_path) and os.path.exists(labels_path):
            if verbose:
                print(f"Processing subdirectory: {subdir}")

            try:  # ATTENTION: any error occurring in this try-catch means that the corresponding IOUs are evaluated as ZERO

                # Open images
                rgb_image = Image.open(rgb_path)
                rgb_array = np.asarray(rgb_image).copy()
                labels_image = Image.open(labels_path).copy()
                labels_array = np.asarray(labels_image)
                if verbose:
                    print(f"  Loaded {rgb_path} and {labels_path}")

                # Run YOUR PREDICT FUNCTION
                predicted_labels_array = predict(model, rgb_array)

                # Evaluate the IOU metric
                IOUs[i,:] = compute_all_iou(labels_array, predicted_labels_array)

                if verbose:
                    labels_vals = np.unique(np.asarray(labels_image))
                    print(f"  Unique labels values: {labels_vals}")
                    predicted_labels_vals = np.unique(np.asarray(predicted_labels_array))
                    print(f"  Unique predicted labels values: {predicted_labels_vals}")

                    plt.subplot(1, 3, 1)
                    plt.imshow(rgb_image)
                    plt.subplot(1, 3, 2)
                    plt.imshow(labels_image)
                    plt.subplot(1, 3, 3)
                    plt.imshow(predicted_labels_array)
                    plt.show()

                rgb_image.close()
                labels_image.close()

            except FileNotFoundError:
                print(f"  Error: Could not find image files in {subdir_path}")
            except Exception as e:
                print(f"  Error processing images in {subdir_path}: {e}")
        else:
            print(f"  Skipping subdirectory {subdir}: rgb.jpg or labels.png not found.")

score = np.nanmean(IOUs)
print(f"\nFinal competition score: {score}")


Modello caricato da model_segmentation.pth e pronto all'uso.


Processing samples: 932it [10:50,  1.43it/s]


Final competition score: 0.0
